> ## SOLUTION / ANSWER KEY &mdash; Lab 3.7
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-07-semantic-search.ipynb`](../lab-07-semantic-search.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 3.7 &mdash; Semantic Search with Real Embeddings

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 3 &mdash; Why Transformers?**

### What you'll do
- Vectorise a corpus with a real sentence-embedding model
- Rank documents by cosine similarity to a query
- Return the top-k most relevant documents with scores

> **How this lab works (near-real):** these labs run **real Hugging Face Transformers** locally on CPU. Read the **Concept**, fill the real `___` blanks in **Build it** (real tokenizer / model / decoding calls), **Run it for real** to see the **actual model output**, note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real model output you can read. The genuine maths (attention, positional encoding, cosine) you still compute **by hand** &mdash; that is real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased` (tokenizer / fill-mask), `sentence-transformers/all-MiniLM-L6-v2` (embeddings), `prajjwal1/bert-tiny` (attention), `distilgpt2` (generation). First use downloads the weights (needs network), then they are cached. The hosted "GPT API" path uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 3 slides &mdash; Tokens & embeddings](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 3 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (used by the text-gen lab)

WORK = "/tmp/biaa-lab-03-07"
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
**Retrieval** powers search and RAG (Day 3): embed every document, embed the query, and return the
closest documents by cosine similarity. Unlike keyword search, this matches **meaning** &mdash; a query
about "a cute kitten" finds a doc about cats even with no shared words. We use **real**
all-MiniLM-L6-v2 embeddings (unit vectors, so a dot product *is* the cosine).

## Build it
`embed()` (given) is real. Complete the query embedding, scoring, and top-k selection.

In [ ]:
import numpy as np
_EMB = {}
def embed(texts):
    """Real sentence embeddings from all-MiniLM-L6-v2: model forward pass -> mean-pool -> unit vector."""
    import torch
    from transformers import AutoTokenizer, AutoModel
    if not _EMB:
        name = "sentence-transformers/all-MiniLM-L6-v2"
        _EMB["tok"] = AutoTokenizer.from_pretrained(name)
        _EMB["mdl"] = AutoModel.from_pretrained(name); _EMB["mdl"].eval()
    if isinstance(texts, str): texts = [texts]
    enc = _EMB["tok"](texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad(): out = _EMB["mdl"](**enc)
    mask = enc["attention_mask"].unsqueeze(-1).float()
    pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1)     # mean over REAL tokens
    return torch.nn.functional.normalize(pooled, dim=1).numpy()      # unit vectors -> dot == cosine

DOCS = ["the cat is a small furry pet animal",
        "dogs are loyal pets that bark",
        "python is a popular programming language",
        "neural networks learn patterns from data",
        "transformers use attention to model language",
        "kittens are playful baby cats"]

def search(query, DOC_E, k=2):
    qv = embed([query])[0]
    sims = DOC_E @ qv
    order = np.argsort(sims)[::-1][:k]
    return [(round(float(sims[i]), 3), DOCS[i]) for i in order]

## Run it for real
Embed the corpus once, then run real queries against it.

In [ ]:
try:
    DOC_E = embed(DOCS)                     # real model -> one unit vector per document
    for query in ["a cute little kitten", "writing software code", "how do transformers work"]:
        print("QUERY:", query)
        for score, doc in search(query, DOC_E, k=2):
            print(f"   {score}  {doc}")
        print()
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

## What to notice
- "a cute little kitten" retrieves the **cat / kitten** docs first &mdash; with **zero shared words**. That is semantic (not keyword) matching.
- Scores are real cosines: a clear top hit scores high; an off-topic query scores low across the board.
- Embed the corpus **once**, reuse for every query &mdash; exactly how a vector database / RAG index works.

## Your turn (open task &mdash; no grader)
Add your own documents and queries. Find a query where semantic search **beats** keyword search
(no shared words but the right hit) and one where it **struggles** (needs an exact term). Try `k=3`.
A "good" answer: you can explain, from the scores, why a particular doc ranked where it did.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
try:
    # search() looks up its results in the module-global DOCS, so extend that same list:
    DOCS = DOCS + ["the ocean is deep and full of fish",
                   "i baked fresh bread this morning"]
    DOC_E = embed(DOCS)                                # re-embed the enlarged corpus
    for query in ["something to eat for dinner", "creatures of the sea"]:
        print("QUERY:", query)
        for score, doc in search(query, DOC_E, k=3):   # top-3 this time
            print(f"   {score}  {doc}")
        print()
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

---
### Key takeaway
Real embeddings + cosine ranking = modern semantic search, and the retrieval half of RAG (which returns on Day 3).

[&#8592; All Module 3 labs](./index.html) &nbsp;&middot;&nbsp; [Module 3 slides](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>